<a href="https://colab.research.google.com/github/juborduchi/governanca-digital_mre/blob/main/notebooks/analise_nota_mre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importação de bibliotecas

In [4]:
import os
import json
import pandas as pd
import glob


# Acesso ao drive

In [5]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


# Organização dos JSONs no DataFrame

In [6]:
def consolidar_jsons_aninhados(caminho_diretorio):
    lista_dataframes = []

    # Listar arquivos
    arquivos = [f for f in os.listdir(caminho_diretorio) if f.endswith('.json')]

    if not arquivos:
        print("Nenhum arquivo JSON encontrado.")
        return None

    for arquivo in arquivos:
        caminho_completo = os.path.join(caminho_diretorio, arquivo)

        try:
            with open(caminho_completo, 'r', encoding='utf-8') as f:
                dados = json.load(f)

                # EXPLICAÇÃO TÉCNICA:
                # Se o seu JSON tem o formato {'_default': {'0': {...}, '1': {...}}},
                # precisamos extrair os valores da chave '_default'.
                if '_default' in dados:
                    # Transformamos os valores de '_default' em uma lista e normalizamos
                    conteudo = list(dados['_default'].values())
                    df_temp = pd.json_normalize(conteudo)
                else:
                    # Caso o JSON seja uma lista direta ou outra estrutura
                    df_temp = pd.json_normalize(dados)

                lista_dataframes.append(df_temp)
        except Exception as e:
            print(f"Erro ao processar {arquivo}: {e}")

    # Concatenar todos os arquivos processados
    df_final = pd.concat(lista_dataframes, ignore_index=True)

    # Limpeza de listas: Se os dados vierem como ['Brasil'], removemos os parênteses
    # Aplicamos uma função que extrai o primeiro elemento se o dado for uma lista
    df_final = df_final.applymap(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x)

    return df_final

# --- CONFIGURAÇÃO DO CAMINHO ---
caminho_pasta = '/content/gdrive/MyDrive/Colab_Notebooks/pibic-2025-2026/json-notas/'

# Execução
df = consolidar_jsons_aninhados(caminho_pasta)

# Exibição do resultado
if df is not None:
    print("DataFrame Organizado:")
    display(df.head())

DataFrame Organizado:


/tmp/ipykernel_3343/1430405251.py:38: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_final = df_final.applymap(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x)


,tipo_dado,pais,origem,sigla,classificado,categoria,autoria,titulo,subtitulo,data,...,nome_arquivo,dir_arquivo,dir_base,codigo_bd,dir_bd,nome_arq_bd,env_dir_bd,extra_01,extra_02,extra_03
0,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Calendário de eventos entre 11 e 19 de janeiro...,,10/01/2014,...,NA,NA,/hdvm12,bd/003/001/001/001/002/001,/json/,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA-...,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,NA,NA,NA
1,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Conferência Internacional sobre Síria (Genebra...,,08/01/2014,...,NA,NA,/hdvm12,bd/003/001/001/001/002/001,/json/,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA-...,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,NA,NA,NA
2,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Calendário de eventos entre 4 de janeiro e 12 ...,,03/01/2014,...,NA,NA,/hdvm12,bd/003/001/001/001/002/001,/json/,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA-...,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,NA,NA,NA
3,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Atentados no Líbano,,02/01/2014,...,NA,NA,/hdvm12,bd/003/001/001/001/002/001,/json/,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA-...,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,NA,NA,NA
4,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Calendário de eventos entre 1º e 9 de fevereir...,,31/01/2014,...,NA,NA,/hdvm12,bd/003/001/001/001/002/001,/json/,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA-...,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,NA,NA,NA


In [7]:
df.columns

Index(['tipo_dado', 'pais', 'origem', 'sigla', 'classificado', 'categoria',
       'autoria', 'titulo', 'subtitulo', 'data', 'horario', 'datatime',
       'data_atualizado', 'horario_atualizado', 'link', 'link_archive',
       'data_archive', 'horario_archive', 'local', 'tags', 'paragrafos',
       'imagens', 'nome_arquivo', 'dir_arquivo', 'dir_base', 'codigo_bd',
       'dir_bd', 'nome_arq_bd', 'env_dir_bd', 'extra_01', 'extra_02',
       'extra_03'],
      dtype='object')

# Preparação dos dados para análise

In [8]:
import pandas as pd
import matplotlib.pyplot as plt

# 1.1 — Converter data para datetime (dayfirst pq é formato brasileiro: dd/mm/aaaa)
df['data_dt'] = pd.to_datetime(df['data'], dayfirst=True, errors='coerce')

# 1.2 — Juntar os parágrafos (que são listas) em um único texto
df['texto'] = df['paragrafos'].apply(lambda lista: ' '.join(lista) if isinstance(lista, list) else str(lista))

# 1.3 — Coluna com tudo junto e em minúsculas (facilita buscas)
df['texto_completo'] = (df['titulo'].fillna('') + ' ' + df['texto']).str.lower()

# 1.4 — Extrair a hora como número inteiro (de "19h20" tira o 19)
df['hora_int'] = df['horario'].str.extract(r'(\d{1,2})').astype(float)

# 1.5 — Colunas auxiliares de tempo
df['ano_mes'] = df['data_dt'].dt.to_period('M')
df['dia_semana'] = df['data_dt'].dt.day_name()
df['tamanho_texto'] = df['texto'].str.len()
df['qtd_paragrafos'] = df['paragrafos'].apply(lambda x: len(x) if isinstance(x, list) else 0)

print(f"Total de notas: {len(df)}")
print(f"Período: de {df['data_dt'].min().date()} até {df['data_dt'].max().date()}")
df[['titulo', 'data_dt', 'hora_int', 'qtd_paragrafos', 'tamanho_texto']].head()

Total de notas: 5169
Período: de 2014-01-02 até 2025-12-31


,titulo,data_dt,hora_int,qtd_paragrafos,tamanho_texto
0,Calendário de eventos entre 11 e 19 de janeiro...,2014-01-10,17.0,0,244
1,Conferência Internacional sobre Síria (Genebra...,2014-01-08,17.0,0,132
2,Calendário de eventos entre 4 de janeiro e 12 ...,2014-01-03,18.0,0,310
3,Atentados no Líbano,2014-01-02,10.0,0,80
4,Calendário de eventos entre 1º e 9 de fevereir...,2014-01-31,18.0,0,85


In [9]:
df_customizado = df[["autoria","titulo",
       "data_atualizado", "horario_atualizado", "link", "paragrafos", "texto_completo","data_dt", "hora_int", "qtd_paragrafos", "tamanho_texto"]]
df_customizado

,autoria,titulo,data_atualizado,horario_atualizado,link,paragrafos,texto_completo,data_dt,hora_int,qtd_paragrafos,tamanho_texto
0,NA,Calendário de eventos entre 11 e 19 de janeiro...,31/10/2022,17h22,https://www.gov.br/mre/pt-br/canais_atendiment...,"9 a 18/JAN – Nova York, EUA. Convenção Anual d...",calendário de eventos entre 11 e 19 de janeiro...,2014-01-10,17.0,0,244
1,NA,Conferência Internacional sobre Síria (Genebra...,31/10/2022,17h21,https://www.gov.br/mre/pt-br/canais_atendiment...,Os profissionais de imprensa interessados em p...,conferência internacional sobre síria (genebra...,2014-01-08,17.0,0,132
2,NA,Calendário de eventos entre 4 de janeiro e 12 ...,31/10/2022,17h21,https://www.gov.br/mre/pt-br/canais_atendiment...,4 a 11/JAN – Singapura. Visita a Singapura do ...,calendário de eventos entre 4 de janeiro e 12 ...,2014-01-03,18.0,0,310
3,NA,Atentados no Líbano,31/10/2022,17h21,https://www.gov.br/mre/pt-br/canais_atendiment...,O Governo brasileiro condena a série recente d...,atentados no líbano o governo brasileiro conde...,2014-01-02,10.0,0,80
4,NA,Calendário de eventos entre 1º e 9 de fevereir...,31/10/2022,17h22,https://www.gov.br/mre/pt-br/canais_atendiment...,"1º e 2/FEV – Melbourne, Austrália. Festival Mu...",calendário de eventos entre 1º e 9 de fevereir...,2014-01-31,18.0,0,85
...,...,...,...,...,...,...,...,...,...,...,...
5164,Ministério das Relações Exteriores,Acordo entre o Reino do Bahrein e o Estado de ...,NA,NA,https://www.gov.br/mre/pt-br/canais_atendiment...,O governo brasileiro recebeu com satisfação ...,acordo entre o reino do bahrein e o estado de ...,2020-09-12,18.0,0,257
5165,Ministério das Relações Exteriores,Declaração Conjunta sobre o Comércio de Etanol...,NA,NA,https://www.gov.br/mre/pt-br/canais_atendiment...,Brasil e Estados Unidos realizaram consultas s...,declaração conjunta sobre o comércio de etanol...,2020-09-11,23.0,0,890
5166,Ministério das Relações Exteriores,Representantes do regime ilegítimo da Venezuel...,NA,NA,https://www.gov.br/mre/pt-br/canais_atendiment...,"Na data de hoje, 4 de setembro de 2020, o Gove...",representantes do regime ilegítimo da venezuel...,2020-09-04,20.0,0,210
5167,Ministério das Relações Exteriores,Embargo indevido do Governo das Filipinas à ca...,NA,NA,https://www.gov.br/mre/pt-br/canais_atendiment...,"O governo da República das Filipinas impôs, ...",embargo indevido do governo das filipinas à ca...,2020-09-04,18.0,0,239


In [10]:
df.shape

(5169, 40)

In [11]:
df_customizado.describe(include="all")

,autoria,titulo,data_atualizado,horario_atualizado,link,paragrafos,texto_completo,data_dt,hora_int,qtd_paragrafos,tamanho_texto
count,5169,5169,5169,5169,5169,5169,5169,5169,5169.000000,5169.0,5169.000000
unique,2,4922,659,669,5169,4975,5156,NaN,NaN,NaN,NaN
top,Ministério das Relações Exteriores,Situação em Israel e na Palestina. Repatriação...,NA,NA,https://www.gov.br/mre/pt-br/canais_atendiment...,.,comunicado do grupo de lima tradução não oficial,NaN,NaN,NaN,NaN
freq,2873,14,2942,2942,1,26,3,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-02-29 02:14:33.360418048,16.212033,0.0,327.447088
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014-01-02 00:00:00,0.000000,0.0,1.000000
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016-08-25 00:00:00,13.000000,0.0,172.000000
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019-06-26 00:00:00,17.000000,0.0,245.000000
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-12-08 00:00:00,20.000000,0.0,359.000000
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-12-31 00:00:00,23.000000,0.0,57109.000000


In [12]:
notas_por_dia = df['data_dt'].value_counts().sort_index()

display(notas_por_dia)

,count
data_dt,
2014-01-02,1
2014-01-03,1
2014-01-08,1
2014-01-10,2
2014-01-14,3
...,...
2025-12-22,2
2025-12-24,1
2025-12-26,1


## Verificando duplicatas

In [13]:
# Duplicatas por link (cada nota deveria ter um link único)
print("Duplicatas por link:", df.duplicated(subset=['link']).sum())

# Duplicatas por título
print("Duplicatas por título:", df.duplicated(subset=['titulo']).sum())




Duplicatas por link: 0
Duplicatas por título: 247


In [14]:

print("\n--- Registros com Títulos Duplicados ---")
duplicadas_titulo = df[df.duplicated(subset=['titulo'], keep=False)]
print(f"Total de linhas envolvidas em duplicatas de título: {len(duplicadas_titulo)}")
display(duplicadas_titulo.sort_values(by='titulo'))


--- Registros com Títulos Duplicados ---
Total de linhas envolvidas em duplicatas de título: 366


,tipo_dado,pais,origem,sigla,classificado,categoria,autoria,titulo,subtitulo,data,...,extra_02,extra_03,data_dt,texto,texto_completo,hora_int,ano_mes,dia_semana,tamanho_texto,qtd_paragrafos
812,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,21ª Conferência das Partes da Convenção-Quadro...,,09/11/2015,...,NA,NA,2015-11-09,"Foi antecipado, para hoje, dia 9 de novembro, ...",21ª conferência das partes da convenção-quadro...,13.0,2015-11,Monday,362,0
760,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,21ª Conferência das Partes da Convenção-Quadro...,,27/10/2015,...,NA,NA,2015-10-27,Encerra-se no próximo dia 20 de novembro o pra...,21ª conferência das partes da convenção-quadro...,18.0,2015-10,Tuesday,326,0
4090,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],Ministério das Relações Exteriores,Abertura de mercado no Peru - Nota Conjunta MR...,,17/04/2025,...,NA,NA,2025-04-17,O governo brasileiro e o governo peruano concl...,abertura de mercado no peru - nota conjunta mr...,17.0,2025-04,Thursday,165,0
4100,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],Ministério das Relações Exteriores,Abertura de mercado no Peru - Nota Conjunta MR...,,08/04/2025,...,NA,NA,2025-04-08,O governo brasileiro e o governo peruano concl...,abertura de mercado no peru - nota conjunta mr...,13.0,2025-04,Tuesday,189,0
4651,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],Ministério das Relações Exteriores,Acidente aéreo na Indonésia,,29/10/2018,...,NA,NA,2018-10-29,O governo brasileiro manifesta seu profundo pe...,acidente aéreo na indonésia o governo brasilei...,22.0,2018-10,Monday,223,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Visita da Comissão de Chanceleres da UNASUL à ...,,05/03/2015,...,NA,NA,2015-03-05,"O Ministro das Relações Exteriores, Embaixador...",visita da comissão de chanceleres da unasul à ...,18.0,2015-03,Thursday,344,0
3745,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],Ministério das Relações Exteriores,Visita do Ministro Mauro Vieira a Omã – 8 de s...,,07/09/2024,...,NA,NA,2024-09-07,"O Ministro das Relações Exteriores, Mauro Vi...",visita do ministro mauro vieira a omã – 8 de s...,15.0,2024-09,Saturday,519,0
3744,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],Ministério das Relações Exteriores,Visita do Ministro Mauro Vieira a Omã – 8 de s...,,08/09/2024,...,NA,NA,2024-09-08,"O Ministro das Relações Exteriores, Mauro Vi...",visita do ministro mauro vieira a omã – 8 de s...,8.0,2024-09,Sunday,519,0
1606,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Visita do ministro Aloysio Nunes Ferreira ao C...,,10/04/2017,...,NA,NA,2017-04-10,"O ministro Aloysio Nunes Ferreira realiza, nos...",visita do ministro aloysio nunes ferreira ao c...,16.0,2017-04,Monday,107,0


### Verificando duplicata por titulo

In [15]:
# Supondo que seu DataFrame seja 'df'
coluna_busca = 'titulo' # ou 'link'

# 1. Contagem de "Excedentes" (O que o seu .sum() estava fazendo)
total_excedentes = df.duplicated(subset=[coluna_busca], keep='first').sum()

# 2. DataFrame de Visualização (Todas as ocorrências)
df_todas_duplicatas = df[df.duplicated(subset=[coluna_busca], keep=False)]
total_na_visualizacao = len(df_todas_duplicatas)

print(f"--- Relatório de Consistência: {coluna_busca} ---")
print(f"Quantidade de cópias excedentes (sum): {total_excedentes}")
print(f"Total de linhas que são duplicatas entre si: {total_na_visualizacao}")
print("-" * 40)

# 3. Exibição Estruturada
# Ordenamos para que você veja os grupos de duplicatas lado a lado
if not df_todas_duplicatas.empty:
    print(f"Listando as duplicatas de {coluna_busca}:")
    display(df_todas_duplicatas.sort_values(by=coluna_busca))
else:
    print("Nenhuma duplicata encontrada com os parâmetros atuais.")


--- Relatório de Consistência: titulo ---
Quantidade de cópias excedentes (sum): 247
Total de linhas que são duplicatas entre si: 366
----------------------------------------
Listando as duplicatas de titulo:


,tipo_dado,pais,origem,sigla,classificado,categoria,autoria,titulo,subtitulo,data,...,extra_02,extra_03,data_dt,texto,texto_completo,hora_int,ano_mes,dia_semana,tamanho_texto,qtd_paragrafos
812,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,21ª Conferência das Partes da Convenção-Quadro...,,09/11/2015,...,NA,NA,2015-11-09,"Foi antecipado, para hoje, dia 9 de novembro, ...",21ª conferência das partes da convenção-quadro...,13.0,2015-11,Monday,362,0
760,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,21ª Conferência das Partes da Convenção-Quadro...,,27/10/2015,...,NA,NA,2015-10-27,Encerra-se no próximo dia 20 de novembro o pra...,21ª conferência das partes da convenção-quadro...,18.0,2015-10,Tuesday,326,0
4090,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],Ministério das Relações Exteriores,Abertura de mercado no Peru - Nota Conjunta MR...,,17/04/2025,...,NA,NA,2025-04-17,O governo brasileiro e o governo peruano concl...,abertura de mercado no peru - nota conjunta mr...,17.0,2025-04,Thursday,165,0
4100,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],Ministério das Relações Exteriores,Abertura de mercado no Peru - Nota Conjunta MR...,,08/04/2025,...,NA,NA,2025-04-08,O governo brasileiro e o governo peruano concl...,abertura de mercado no peru - nota conjunta mr...,13.0,2025-04,Tuesday,189,0
4651,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],Ministério das Relações Exteriores,Acidente aéreo na Indonésia,,29/10/2018,...,NA,NA,2018-10-29,O governo brasileiro manifesta seu profundo pe...,acidente aéreo na indonésia o governo brasilei...,22.0,2018-10,Monday,223,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Visita da Comissão de Chanceleres da UNASUL à ...,,05/03/2015,...,NA,NA,2015-03-05,"O Ministro das Relações Exteriores, Embaixador...",visita da comissão de chanceleres da unasul à ...,18.0,2015-03,Thursday,344,0
3745,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],Ministério das Relações Exteriores,Visita do Ministro Mauro Vieira a Omã – 8 de s...,,07/09/2024,...,NA,NA,2024-09-07,"O Ministro das Relações Exteriores, Mauro Vi...",visita do ministro mauro vieira a omã – 8 de s...,15.0,2024-09,Saturday,519,0
3744,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],Ministério das Relações Exteriores,Visita do Ministro Mauro Vieira a Omã – 8 de s...,,08/09/2024,...,NA,NA,2024-09-08,"O Ministro das Relações Exteriores, Mauro Vi...",visita do ministro mauro vieira a omã – 8 de s...,8.0,2024-09,Sunday,519,0
1606,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Visita do ministro Aloysio Nunes Ferreira ao C...,,10/04/2017,...,NA,NA,2017-04-10,"O ministro Aloysio Nunes Ferreira realiza, nos...",visita do ministro aloysio nunes ferreira ao c...,16.0,2017-04,Monday,107,0


In [16]:
df_teste = df_todas_duplicatas[["titulo","link"]]
df_teste

,titulo,link
43,Situação na Ucrânia,https://www.gov.br/mre/pt-br/canais_atendiment...
86,Atentado na Nigéria,https://www.gov.br/mre/pt-br/canais_atendiment...
147,Concessão de agrément ao Embaixador do Brasil ...,https://www.gov.br/mre/pt-br/canais_atendiment...
153,Situação na Síria,https://www.gov.br/mre/pt-br/canais_atendiment...
190,Eleições no Equador,https://www.gov.br/mre/pt-br/canais_atendiment...
...,...,...
5106,Eleições na Guiana,https://www.gov.br/mre/pt-br/canais_atendiment...
5122,Comunicado do Grupo de Lima,https://www.gov.br/mre/pt-br/canais_atendiment...
5130,Nota Conjunta do Ministério das Relações Exter...,https://www.gov.br/mre/pt-br/canais_atendiment...
5141,Situação no Mali,https://www.gov.br/mre/pt-br/canais_atendiment...


[Tabela duplicatas por titulo](https://docs.google.com/spreadsheets/d/1BfJQLx3d--ejWnkmRbHOJFZRgvkRLnr9/edit?gid=585304390#gid=585304390)

### Verificando duplicata por texto

In [17]:
# Supondo que seu DataFrame seja 'df'
coluna_busca = 'texto_completo' # ou 'link'

# 1. Contagem de "Excedentes" (O que o seu .sum() estava fazendo)
duplicatas_texto = df.duplicated(subset=[coluna_busca], keep='first').sum()

# 2. DataFrame de Visualização (Todas as ocorrências)
df_duplicatas_texto = df[df.duplicated(subset=[coluna_busca], keep=False)]
total_na_visualizacao = len(df_duplicatas_texto)

print(f"--- Relatório de Consistência: {coluna_busca} ---")
print(f"Quantidade de cópias excedentes (sum): {duplicatas_texto}")
print(f"Total de linhas que são duplicatas entre si: {total_na_visualizacao}")
print("-" * 40)

# 3. Exibição Estruturada
# Ordenamos para que você veja os grupos de duplicatas lado a lado
if not df_duplicatas_texto.empty:
    print(f"Listando as duplicatas de {coluna_busca}:")
    display(df_duplicatas_texto.sort_values(by=coluna_busca))
else:
    print("Nenhuma duplicata encontrada com os parâmetros atuais.")

--- Relatório de Consistência: texto_completo ---
Quantidade de cópias excedentes (sum): 13
Total de linhas que são duplicatas entre si: 24
----------------------------------------
Listando as duplicatas de texto_completo:


,tipo_dado,pais,origem,sigla,classificado,categoria,autoria,titulo,subtitulo,data,...,extra_02,extra_03,data_dt,texto,texto_completo,hora_int,ano_mes,dia_semana,tamanho_texto,qtd_paragrafos
2158,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Comunicado do Grupo de Lima,,14/05/2018,...,NA,NA,2018-05-14,Tradução não oficial,comunicado do grupo de lima tradução não oficial,22.0,2018-05,Monday,20,0
2148,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Comunicado do Grupo de Lima,,18/05/2018,...,NA,NA,2018-05-18,Tradução não oficial,comunicado do grupo de lima tradução não oficial,22.0,2018-05,Friday,20,0
2103,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Comunicado do Grupo de Lima,,07/04/2018,...,NA,NA,2018-04-07,Tradução não oficial,comunicado do grupo de lima tradução não oficial,16.0,2018-04,Saturday,20,0
1624,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Comunicado sobre a Venezuela,,30/04/2017,...,NA,NA,2017-04-30,Comunicado sobre a Venezuela,comunicado sobre a venezuela comunicado sobre ...,23.0,2017-04,Sunday,28,0
1602,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Comunicado sobre a Venezuela,,17/04/2017,...,NA,NA,2017-04-17,Comunicado sobre a Venezuela,comunicado sobre a venezuela comunicado sobre ...,23.0,2017-04,Monday,28,0
441,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Concessão de agrément ao Embaixador do Brasil ...,,20/03/2015,...,NA,NA,2015-03-20,Concessão de agrément ao Embaixador do Brasil ...,concessão de agrément ao embaixador do brasil ...,22.0,2015-03,Friday,59,0
886,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Concessão de agrément ao Embaixador do Brasil ...,,20/01/2016,...,NA,NA,2016-01-20,Concessão de agrément ao Embaixador do Brasil ...,concessão de agrément ao embaixador do brasil ...,14.0,2016-01,Wednesday,59,0
4621,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Declaração do Grupo de Lima,,13/08/2018,...,NA,NA,2018-08-13,Tradução não oficial,declaração do grupo de lima tradução não oficial,12.0,2018-08,Monday,20,0
2032,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Declaração do Grupo de Lima,,14/02/2018,...,NA,NA,2018-02-14,Tradução não oficial,declaração do grupo de lima tradução não oficial,11.0,2018-02,Wednesday,20,0
4575,aberto,Brasil,ministerio das relações exteriores,BD_DE_LATAM_BRA_GOVFEDERAL_MRE_NOTAS_IMPRENSA,noticias institucionais,[],NA,Declaração do Grupo de Lima,,15/09/2018,...,NA,NA,2018-09-15,Tradução não oficial,declaração do grupo de lima tradução não oficial,22.0,2018-09,Saturday,20,0


In [18]:
teste = df_duplicatas_texto[["titulo","texto_completo","link"]]
teste

,titulo,texto_completo,link
441,Concessão de agrément ao Embaixador do Brasil ...,concessão de agrément ao embaixador do brasil ...,https://www.gov.br/mre/pt-br/canais_atendiment...
635,Eleições no Haiti,eleições no haiti eleições no haiti,https://www.gov.br/mre/pt-br/canais_atendiment...
886,Concessão de agrément ao Embaixador do Brasil ...,concessão de agrément ao embaixador do brasil ...,https://www.gov.br/mre/pt-br/canais_atendiment...
916,Eleições no Haiti,eleições no haiti eleições no haiti,https://www.gov.br/mre/pt-br/canais_atendiment...
1601,Visita do ministro Aloysio Nunes Ferreira ao C...,visita do ministro aloysio nunes ferreira ao c...,https://www.gov.br/mre/pt-br/canais_atendiment...
1602,Comunicado sobre a Venezuela,comunicado sobre a venezuela comunicado sobre ...,https://www.gov.br/mre/pt-br/canais_atendiment...
1606,Visita do ministro Aloysio Nunes Ferreira ao C...,visita do ministro aloysio nunes ferreira ao c...,https://www.gov.br/mre/pt-br/canais_atendiment...
1624,Comunicado sobre a Venezuela,comunicado sobre a venezuela comunicado sobre ...,https://www.gov.br/mre/pt-br/canais_atendiment...
2032,Declaração do Grupo de Lima,declaração do grupo de lima tradução não oficial,https://www.gov.br/mre/pt-br/canais_atendiment...
2103,Comunicado do Grupo de Lima,comunicado do grupo de lima tradução não oficial,https://www.gov.br/mre/pt-br/canais_atendiment...


# Análise e vizualiação de dados

## Volume de publicações

In [19]:
# Extrair apenas o ano
df["ano"] = df["data_dt"].dt.year

notas_por_ano = df['ano'].value_counts().sort_index()

display(notas_por_ano)

,count
ano,
2014,343
2015,536
2016,614
2017,499
2018,427
2019,318
2020,172
2021,186
2022,200
